# Parte 2: Búsquedas de Microsoft Azure AI Search
A continuación vamos a realizar las conexiones al buscador y a OpenAI para llevar a cabo los 4 tipos de búsqueda obligatorios que se piden en la entrega.

In [1]:
!pip install azure-search-documents azure-identity openai python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Credenciales de Azure AI Search
search_endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
search_key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

# Credenciales de Embeddings de Azure OpenAI (para vectorizar nuestra pregunta)
openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
openai_api_key = os.getenv("AZURE_OPENAI_KEY")
openai_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

### Preparar los Clientes

In [3]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizableTextQuery
from openai import AzureOpenAI

# Iniciamos el cliente de Búsqueda
search_client = SearchClient(search_endpoint, index_name, AzureKeyCredential(search_key))

# Iniciamos el cliente de OpenAI para traducir nuestra pregunta si es necesario
openai_client = AzureOpenAI(
    api_key=openai_api_key,
    azure_endpoint=openai_endpoint,
    api_version="2024-02-15-preview"
)

def obtener_embedding(texto):
    response = openai_client.embeddings.create(input=texto, model=openai_deployment)
    return response.data[0].embedding

query = "Resumen del tema principal de mis documentos"

In [4]:
def printFormateado(results, titulo):
    print(f"\n{'='*80}")
    print(f"{titulo.upper()} ")
    print(f"{'='*80}")
    
    count = 0
    for result in results:
        count += 1
        score = result.get('@search.reranker_score') or result.get('@search.score')
        print(f"\n TOP {count} | Relevancia: {score:.4f}")
        print(f" -- Doc: {result.get('title', 'Sin título')}")
        
        # Limpieza de texto para el output
        texto = result.get('chunk', '').replace('\n', ' ').strip()
        print(f" --- Fragmento: {texto[:250]}...")
        print(f"{'·'*80}")
    
    if count == 0:
        print(" No se han encontrado resultados.")

# Definimos la pregunta genérica para todas las pruebas
query_text = "¿Cuál es el resumen principal de los documentos?"
query_vector = obtener_embedding(query_text)
semantic_config = "rag-1776325358245-semantic-configuration"


## 1. Vector Search

In [5]:
res1 = search_client.search(
    search_text=None,
    vector_queries=[{
        "vector": query_vector,
        "k_nearest_neighbors": 5,
        "fields": "text_vector",
        "kind": "vector"
    }],
    select=["title", "chunk"],
    top=5
)
printFormateado(res1, "Búsqueda Vectorial (Matemática)")



BÚSQUEDA VECTORIAL (MATEMÁTICA) 

 TOP 1 | Relevancia: 0.8026
 -- Doc: TEORIA.md
 --- Fragmento: Azure AI Search** | What is Azure AI Search? | [Microsoft Learn](https://learn.microsoft.com/en-us/azure/search/search-what-is-azure-search) | | **🔍 Vector Search** | Vector Search Overview | [Microsoft Learn](https://learn.microsoft.com/en-us/azure/...
················································································

 TOP 2 | Relevancia: 0.7981
 -- Doc: TEORIA.md
 --- Fragmento: # 🎯 Embeddings y Vector Database (Azure AI Search)  > **📖 Resumen:** Aprende a transformar texto en vectores con Azure OpenAI y almacenarlos en Azure AI Search para realizar búsquedas semánticas inteligentes. Fundamentos esenciales para implementar R...
················································································

 TOP 3 | Relevancia: 0.7977
 -- Doc: TEORIA.md
 --- Fragmento: que los embeddings resuelven de forma elegante.  ### 🔴 El Problema de la Búsqueda Tradicional  Imagina q

## 2. Hybrid Search (Keyword + Vector)

In [6]:
res2 = search_client.search(
    search_text=query_text,
    vector_queries=[{
        "vector": query_vector,
        "k_nearest_neighbors": 5,
        "fields": "text_vector",
        "kind": "vector"
    }],
    select=["title", "chunk"],
    top=5
)
printFormateado(res2, "Búsqueda Híbrida (Texto + Vector)")



BÚSQUEDA HÍBRIDA (TEXTO + VECTOR) 

 TOP 1 | Relevancia: 0.0328
 -- Doc: TEORIA.md
 --- Fragmento: que los embeddings resuelven de forma elegante.  ### 🔴 El Problema de la Búsqueda Tradicional  Imagina que tienes un documento que habla sobre **"perros"** y un usuario busca **"caninos"**. La búsqueda tradicional por palabras clave **NO encontrará e...
················································································

 TOP 2 | Relevancia: 0.0318
 -- Doc: TEORIA.md
 --- Fragmento: # 🎯 Embeddings y Vector Database (Azure AI Search)  > **📖 Resumen:** Aprende a transformar texto en vectores con Azure OpenAI y almacenarlos en Azure AI Search para realizar búsquedas semánticas inteligentes. Fundamentos esenciales para implementar R...
················································································

 TOP 3 | Relevancia: 0.0308
 -- Doc: TEORIA.md
 --- Fragmento: index_name="mi-indice-vectorial",     credential=AzureKeyCredential("TU-API-KEY") )  # Preparar docume

## 3. Semantic Search

In [7]:
res3 = search_client.search(
    search_text=query_text,
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    select=["title", "chunk"],
    top=5
)
printFormateado(res3, "Búsqueda Semántica (Reranker)")



BÚSQUEDA SEMÁNTICA (RERANKER) 

 TOP 1 | Relevancia: 1.7965
 -- Doc: TEORIA.md
 --- Fragmento: # 🎯 Embeddings y Vector Database (Azure AI Search)  > **📖 Resumen:** Aprende a transformar texto en vectores con Azure OpenAI y almacenarlos en Azure AI Search para realizar búsquedas semánticas inteligentes. Fundamentos esenciales para implementar R...
················································································

 TOP 2 | Relevancia: 1.6065
 -- Doc: TEORIA.md
 --- Fragmento: después de RRF )  for result in results:     print(f"🎯 Score híbrido: {result['@search.score']:.4f}")     print(f"📄 {result['title']}\n") ```  **💡 Best Practice:** Para Hybrid Search, usa `k=50` en vectores y `top=10` final. Esto le da más datos al a...
················································································

 TOP 3 | Relevancia: 1.5718
 -- Doc: TEORIA.md
 --- Fragmento: contenido similar basándose en significado |  ---  ## 🔢 2. EMBEDDINGS CON AZURE OPENAI: Lo Esencial  > 💡 *

## 4. Semantic Hybrid Search (Keyword + Vector + Semantic Ranking)

In [8]:
res4 = search_client.search(
    search_text=query_text,
    vector_queries=[{
        "vector": query_vector,
        "k_nearest_neighbors": 5,
        "fields": "text_vector",
        "kind": "vector"
    }],
    query_type="semantic",
    semantic_configuration_name=semantic_config,
    top=5
)
printFormateado(res4, "Búsqueda Híbrida Semántica (Máxima Precisión)")



BÚSQUEDA HÍBRIDA SEMÁNTICA (MÁXIMA PRECISIÓN) 

 TOP 1 | Relevancia: 1.7965
 -- Doc: TEORIA.md
 --- Fragmento: # 🎯 Embeddings y Vector Database (Azure AI Search)  > **📖 Resumen:** Aprende a transformar texto en vectores con Azure OpenAI y almacenarlos en Azure AI Search para realizar búsquedas semánticas inteligentes. Fundamentos esenciales para implementar R...
················································································

 TOP 2 | Relevancia: 1.6065
 -- Doc: TEORIA.md
 --- Fragmento: después de RRF )  for result in results:     print(f"🎯 Score híbrido: {result['@search.score']:.4f}")     print(f"📄 {result['title']}\n") ```  **💡 Best Practice:** Para Hybrid Search, usa `k=50` en vectores y `top=10` final. Esto le da más datos al a...
················································································

 TOP 3 | Relevancia: 1.5718
 -- Doc: TEORIA.md
 --- Fragmento: contenido similar basándose en significado |  ---  ## 🔢 2. EMBEDDINGS CON AZURE OPENAI: Lo